# Hardline Tamil Narration — OmniVoice voice clone (batch)

**Runtime → Change runtime type → GPU (T4)** first. Then run cells top to bottom.

Uses the same install + `model.generate` calls proven to work in the Gradio demo, but batches all 6 Tamil scenes and downloads one `hardline_audio.zip`.
No kernel restart, no HF login, no transcript needed (Whisper auto-transcribes the reference).

After download: unzip the wavs into `hardline-video/public/audio/clone/` locally → `node scripts/build-timeline.mjs`.

In [ ]:
# CELL 1 — install (proven recipe; no restart needed)
!pip install -q omnivoice
!pip install -q torchaudio --extra-index-url https://download.pytorch.org/whl/cu128

import torch, torchaudio
from omnivoice import OmniVoice

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

print('Loading OmniVoice (first time downloads ~3-5 GB)...')
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=dtype)
print('Model loaded!')

In [ ]:
# CELL 2 — upload your reference audio (the ElevenLabs sample .mp3)
from google.colab import files
up = files.upload()
REF_AUDIO = list(up.keys())[0]
print('reference:', REF_AUDIO)

In [ ]:
# CELL 3 — the 6 narration scenes (Tamil). Edit freely.
# REF_TEXT optional: leave "" to let Whisper auto-transcribe the sample.
REF_TEXT = ""
NUM_STEPS = 32   # 16 = faster draft, 32 = final quality
SPEED = 1.0

SCENES = {
  "s1_hook": "தமிழ்நாட்டின் அரசுப் பள்ளிகள் இன்று ஒரு பெரிய கல்வி நெருக்கடியை சந்தித்து வருகின்றன.",
  "s2_overall": "யூடைஸ் பிளஸ் அறிக்கையின்படி, ஒரே ஆண்டில் ஒரு லட்சத்து பதினைந்தாயிரம் மாணவர்கள் பொதுக் கல்வியிலிருந்து குறைந்துள்ளனர். கடந்த இரண்டு ஆண்டுகளில் மட்டும் இந்த சரிவு ஐந்து புள்ளி ஒன்பது லட்சம்.",
  "s3_govt_vs_private": "அரசுப் பள்ளிகள் ஒரே ஆண்டில் இரண்டு லட்சத்து ஏழாயிரம் மாணவர்களை இழந்துள்ளன. ஆனால் அதே நேரத்தில், தனியார் பள்ளிகளில் ஒன்றேமுக்கால் லட்சம் மாணவர்கள் புதிதாக சேர்ந்துள்ளனர்.",
  "s4_share": "இதன் விளைவாக, அரசுப் பள்ளிகளின் பங்கு முப்பத்தொன்பது சதவீதத்திலிருந்து முப்பத்தைந்து சதவீதமாக சுருங்கியுள்ளது. தனியார் பள்ளிகளோ ஐம்பது சதவீதம் என்ற பாதி அளவைத் தொட்டுவிட்டன.",
  "s5_ground": "மதுரை உசிலம்பட்டியில், அறுபத்தைந்து ஆண்டுகளாக அந்த கிராமமே சொந்த உழைப்பில் நடத்திய தொடக்கப் பள்ளி, எஞ்சியிருந்த பத்து மாணவர்களும் தனியார் பள்ளிக்கு சென்றதால், இந்த ஆண்டு நிரந்தரமாக பூட்டப்பட்டுவிட்டது.",
  "s6_outro": "பொதுக் கல்வியின் அடித்தளமே ஆட்டம் கண்டுவருகிறது. முழு செய்தியையும் படிக்க, தி ஹார்ட்லைன்."
}
print(len(SCENES), 'scenes')

In [ ]:
# CELL 4 — generate every scene (same generate() call as the working demo)
import os, json, subprocess

# Convert the uploaded mp3 to a clean 24k mono wav reference.
REF_WAV = 'ref.wav'
subprocess.run(['ffmpeg','-y','-i',REF_AUDIO,'-ar','24000','-ac','1',REF_WAV],
               check=True, capture_output=True)

def synth(text, path):
    kwargs = dict(text=text, ref_audio=REF_WAV, num_step=int(NUM_STEPS), speed=float(SPEED))
    if REF_TEXT.strip():
        kwargs['ref_text'] = REF_TEXT
    audio = model.generate(**kwargs)
    t = audio[0] if isinstance(audio[0], torch.Tensor) else torch.tensor(audio[0])
    if t.dim() == 1:
        t = t.unsqueeze(0)
    torchaudio.save(path, t.cpu().float(), 24000)
    return t.shape[-1] / 24000.0

os.makedirs('out', exist_ok=True)
durations = {}
for sid, text in SCENES.items():
    durations[sid] = round(synth(text, f'out/{sid}.wav'), 3)
    print(sid, durations[sid], 's')

with open('out/durations.json','w') as f:
    json.dump(durations, f, indent=2)
print('total', round(sum(durations.values()),1), 's')

In [ ]:
# CELL 5 — zip + download
import shutil
shutil.make_archive('hardline_audio', 'zip', 'out')
from google.colab import files
files.download('hardline_audio.zip')